# EE 241 · Homework 1 — Basics for Engineering Computation

**Programming for Signal and Information Processing Applications**

**Concepts:** the command line (CLI), your IDE, git, variables, expressions, conditionals, loops
**Application:** numerical signal examples — building and processing a discrete-time signal `x[n]`

---

### How to work this homework
- You can talk through the homework with your lab partner, but work on it and submit it on your own.
- Only tools this week: plain Python and the `math` module. **No NumPy yet** — that's Homework 2. You'll build signals by hand with loops.
- Parts 5 and 6 introduce **agentic AI**. You build the agent yourself, out of the loops you already know.
- Run each cell top to bottom. Finish every `# TODO` in **Parts 4, 5, and 6**, then write the two verification notes at the end.

### Learning outcomes
By the end you can:
1. Run Python from the command line, edit in the IDE, and commit your work with git.
2. Use variables and expressions to represent signal parameters and compute sample values.
3. Use conditionals to make decisions about individual samples (threshold, rectify).
4. Use loops to generate a discrete-time signal one sample at a time and summarize it.
5. Explain what makes a program an **agent** — a goal, a loop, and a stopping rule — and build one.
6. **Verify** code you did not write, including code produced by an AI assistant.

## Part 0 — Set up your tools (CLI · IDE · git)

You'll use three tools every week. Do this once at the start.

**Command line (CLI).** Open a terminal in your IDE and check Python is available:
```bash
python --version
```

**Get the notebook.** Download it from the Canvas assignment and open it in Jupyter.

**The engineering loop.** For the rest of the quarter you'll repeat this loop:
```bash
# edit code in the IDE  ->  run it  ->  check the output  ->  save your work
python lab1.py            # run
git add lab1.py           # stage your change
git commit -m "Homework 1: signal generation working"   # record it
```

Run the cell below to confirm your Python environment works inside the notebook.

In [ ]:
import sys, math
print("Python", sys.version.split()[0])
print("math module loaded, pi =", round(math.pi, 4))

## Part 1 — Variables and expressions

A signal is described by a few **parameters** (numbers you name) and a **formula** (an expression).
Here we describe a simple falling *ramp* signal `x[n] = amplitude - step * n`.

In [ ]:
# Signal parameters (variables)
amplitude = 3.0     # float: peak value
step      = 0.5     # float: how much each sample drops
n         = 2       # int: which sample we want

# One sample, computed from an expression
x_n = amplitude - step * n     # 3.0 - 0.5*2 = 2.0
print("x[", n, "] =", x_n)

**Read the types.** `amplitude` and `step` are *floats* (they have a decimal point); `n` is an *int*.
Watch how division behaves — this is a classic source of bugs:
- `/`  is **float division**: `7 / 2` -> `3.5`
- `//` is **floor division**: `7 // 2` -> `3`

In [ ]:
print("7 / 2  =", 7 / 2)     # 3.5
print("7 // 2 =", 7 // 2)    # 3
print("7 % 2  =", 7 % 2)     # 1  (remainder — you'll use this in Part 4)

## Part 2 — Conditionals: making decisions about a sample

Signal processing constantly asks *"is this sample above a threshold?"* We answer with `if / else`.
Below we **half-wave rectify** or **clip** one sample: keep it positive, otherwise force it to zero.

In [ ]:
sample = -1.5

if sample > 0:
    y = sample
else:
    y = 0.0

print("rectified:", y)      # -1.5 -> 0.0

# Clipping: never let a value exceed a ceiling
ceiling = 2.0
value = 3.7
if value > ceiling:
    value = ceiling
print("clipped:", value)    # 3.7 -> 2.0

## Part 3 — Loops: build a whole signal, one sample at a time

`range(N)` produces the integers `0, 1, ..., N-1` — that's **N numbers, ending at N-1, not N**.
We loop over `n` and `append` each computed sample to a list.

In [ ]:
num_samples = 8
amplitude   = 3.0
step        = 0.5

signal = []                       # start empty
for n in range(num_samples):      # n = 0,1,2,...,7
    signal.append(amplitude - step * n)

print(signal)
print("length:", len(signal))     # 8 samples

## Part 4 — Put it together

Generate a discrete-time **sawtooth-style ramp** that resets every `period` samples, threshold it,
find its maximum, and draw it as a text "stem" plot — using only what you learned above.

Complete every `# TODO`. Each TODO is one or two lines.

In [ ]:
# ---- Parameters ----
period      = 5      # the ramp resets every 5 samples
num_samples = 15
threshold   = 2      # Part 4b uses this

# ---- TODO 1: build the signal `x` ----
# For each n, the sample value is (n % period): 0,1,2,3,4,0,1,2,3,4,...
# Use the modulo operator % inside the loop.
x = []
for n in range(num_samples):
    pass                          # <-- TODO 1: append the value for sample n to x
print("x =", x)

# ---- TODO 2: threshold into `y` ----
# Keep a sample if value >= threshold, otherwise set it to 0. Use if/else.
y = []
for value in x:
    pass                          # <-- TODO 2: append value, or 0, to y
print("y =", y)

# ---- TODO 3: maximum of x WITHOUT the built-in max() ----
# Start x_max at the first sample, then loop over x and update it when you see a larger value.
x_max = None                      # <-- TODO 3: replace None, then write the loop
print("max(x) =", x_max)

# ---- TODO 4: ASCII stem plot of x ----
# Print one row per sample: the index, then a bar of '#' as tall as the value.
print("\nstem plot of x[n]:")
for n in range(num_samples):
    pass                          # <-- TODO 4: print n and '#' * x[n]

### Expected shape (when your TODOs are correct)

`x` should be `[0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4]`, `max(x)` should be `4`, and the stem plot
should show the repeating ramp:

```
0:
1: #
2: ##
3: ###
4: ####
5:
6: #
...
```

## Part 5 — Your first agent

### What "agentic AI" means

Everything you wrote in Parts 1–4 runs **once**, top to bottom. You decided how many steps it took,
and you decided when it was finished.

An **agent** is different in one specific way: it runs in a **loop with a goal**, and *it* decides when
it is finished. Each time around, it looks at its own result and adjusts. That cycle has four stages:

| Stage | What it means | In your code below |
|-------|---------------|--------------------|
| **Perceive** | Measure the current situation | count how many samples survive the threshold |
| **Decide** | Compare the measurement to the goal | is `kept` small enough yet? |
| **Act** | Change something | raise the threshold by 1 |
| **Observe** | Loop again with the new result | back to the top |

This is the same edit → run → check → commit loop from Part 0, except the program runs it instead of you.

### The guard

A loop that decides its own stopping condition can fail to stop. If the goal can never be reached, the
loop runs forever. So every agent gets a **guard** — a hard cap on the number of passes.

Run the worked example below before you attempt the TODOs.

In [1]:
# ---- Worked example: a tiny loop with a goal and a guard ----
# Goal: keep adding 0 + 1 + 2 + ... until the running total reaches 6.

total     = 0
n         = 0
max_steps = 10          # THE GUARD: never more than 10 passes, no matter what

for step in range(max_steps):
    if total >= 6:      # DECIDE: goal reached?
        break           # ...then stop early
    total = total + n   # ACT
    n = n + 1

print("total =", total, "after", n, "steps")   # -> total = 6 after 4 steps

total = 6 after 4 steps


### The task

Your signal `x` from Part 4 is the repeating ramp `[0, 1, 2, 3, 4, 0, 1, ...]`.

Thresholding it keeps only the samples at or above some cutoff. A **low** threshold keeps almost
everything; a **high** threshold keeps almost nothing. Suppose you want to keep **at most 6 samples**
but you don't know what threshold does that.

Rather than guessing, write an agent that finds it: start the threshold at 0 and raise it one step at a
time until few enough samples survive.

Complete the three TODOs. The printed trace is already written for you — it is how you *observe* the
loop converging, and you should read it carefully when you run it.

In [2]:
# ---- Parameters ----
goal_kept = 6      # we want AT MOST this many samples to survive
threshold = 0      # start permissive: at threshold 0, every sample survives
max_steps = 20     # the guard

print("step | threshold | kept")
print("-----+-----------+-----")

for step in range(max_steps):

    # ---- TODO 5a: PERCEIVE ----
    # Count how many samples in x are >= threshold. Use a loop and an if.
    # Start kept at 0 and add 1 for each sample that survives.
    kept = 0
    # your loop here


    print(f"{step:4d} | {threshold:9d} | {kept:4d}")

    # ---- TODO 5b: DECIDE ----
    # If kept is now less than or equal to goal_kept, the goal is met -> break.


    # ---- TODO 5c: ACT ----
    # Otherwise raise the threshold by 1 and go around again.


print()
print("agent settled on threshold =", threshold, "keeping", kept, "samples")

step | threshold | kept
-----+-----------+-----
   0 |         0 |    0
   1 |         0 |    0
   2 |         0 |    0
   3 |         0 |    0
   4 |         0 |    0
   5 |         0 |    0
   6 |         0 |    0
   7 |         0 |    0
   8 |         0 |    0
   9 |         0 |    0
  10 |         0 |    0
  11 |         0 |    0
  12 |         0 |    0
  13 |         0 |    0
  14 |         0 |    0
  15 |         0 |    0
  16 |         0 |    0
  17 |         0 |    0
  18 |         0 |    0
  19 |         0 |    0

agent settled on threshold = 0 keeping 0 samples


### Expected result

When your TODOs are right, the trace converges in four passes and stops on its own:

```
step | threshold | kept
-----+-----------+-----
   0 |         0 |   15
   1 |         1 |   12
   2 |         2 |    9
   3 |         3 |    6

agent settled on threshold = 3 keeping 6 samples
```

The loop stopped at step 3, not step 19. The goal ended it, not the guard.

### TODO 5d — make the guard do its job

Copy your agent into the cell below and change **one thing**: set `goal_kept = -1`.

You can never keep fewer than zero samples, so this goal can never be met. Run it, then answer the
question in the cell underneath.

In [ ]:
# ---- TODO 5d: paste your agent here, with goal_kept = -1 ----
# Then run it and watch what happens.

In [ ]:
# ---- TODO 5d answer ----
guard_answer = """
TODO: The goal was impossible, so what stopped the loop? What was the final
value of threshold, and why that number? In one or two sentences, say what
would have happened if max_steps had not been there.
"""
print(guard_answer)

## Part 6 — Verify the assistant

Later in this course you will use AI assistants to help write code. Starting now, you need the skill
that makes that safe.

AI-written code almost always **runs**. It looks clean and prints a number that looks reasonable. None
of that means it is correct. The failure you have to catch is code that does something subtly different
from what was actually asked.

Build a case small enough to check by hand, then compare.

### The specification

> Count how many samples in a list are **at or above** a cutoff.

"At or above" is the requirement.

### What the assistant produced

The cell below is what an AI assistant returned when given that specification. Run it.

In [ ]:
# ---- Code from an AI assistant. Task: "count samples at or above the cutoff" ----
cutoff = 2
count  = 0

for i in range(len(x)):
    if x[i] > cutoff:
        count = count + 1

print("samples at or above", cutoff, ":", count)

It ran. It printed a number. The number is wrong.

Don't go hunting through `x`. It has 15 samples. Build a case so small the right answer is not in
question.

In [ ]:
# ---- TODO 6a: build a hand-checkable test case ----
# Use this tiny list and cutoff:
small  = [1, 2, 3]
cutoff = 2

# FIRST, by hand and before running anything: which values in `small` are AT OR ABOVE 2?
# Write the count you expect here as a number:
expected = None       # <-- TODO: replace None with the answer you worked out by hand

# NOW run the assistant's code on `small` (copy its loop here, using `small` instead of `x`):
count = 0
# your copy of the assistant's loop here


print("expected:", expected, " assistant gave:", count)

In [ ]:
# ---- TODO 6b: name the bug ----
bug_description = """
TODO: In one sentence, what does the assistant's code actually count, and how
does that differ from what the specification asked for?
"""
print(bug_description)

In [ ]:
# ---- TODO 6c: fix it ----
# Write a corrected version and run it on the full signal x with cutoff = 2.
cutoff = 2
count  = 0
# your corrected loop here


print("corrected count on x:", count)     # should be 9, not 6

### Why this one matters

The bug was a single character. The code never crashed or warned you, and it printed `6` — a plausible
number for a 15-sample signal. Checking it against a case you computed by hand is the only way you
would have caught it.

## Code-reading prompts

Don't run these first — **predict** the output, then run to check. Being able to read code you didn't
write is a core skill this quarter (especially when the code came from an AI assistant).

In [ ]:
# Prompt 1
print(7 // 2, 7 / 2)

In [ ]:
# Prompt 2
for i in range(1, 4):
    print(i, end=" ")

In [ ]:
# Prompt 3
a = 2
b = 3
print(a == b, a < b, a + b == 5)

In [ ]:
# Prompt 4
total = 0
for n in range(4):
    total = total + n
print(total)

## Debugging cases

Each cell has a stated **goal** and a **bug**. Predict what it does, then fix it so it matches the goal.

In [ ]:
# Debug A — Goal: print the samples n = 1, 2, 3, 4, 5
for n in range(1, 5):
    print(n)
# It stops at 4. Fix the range so all five values print.

In [ ]:
# Debug B — Goal: the average of a and b (expected 3.5)
a = 3
b = 4
avg = (a + b) // 2
print(avg)
# It prints 3, not 3.5. Fix the operator.

## Misconception checks

Read each claim and decide **true or false**, then run the cell to see the reality.

In [ ]:
# Claim: "range(4) includes 4."
print(list(range(4)))
# Reality: range(N) gives 0..N-1 -> [0, 1, 2, 3]. It never includes N.

In [ ]:
# Claim: "/ and // give the same answer."
print(9 / 4, 9 // 4)
# Reality: / is float division (2.25); // floors to an int-like value (2).

In [ ]:
# Claim: "assigning a list to a new name makes a separate copy."
a = [1, 2]
b = a
b.append(3)
print("a =", a)
# Reality: a and b are two names for the SAME list -> a is now [1, 2, 3].
# (To truly copy: b = a.copy() or b = list(a).)

In Python, the indented block is the body of the `if`/`for`. Changing the indentation changes what the
code does, not just its appearance.

## Deliverable & verification

1. Complete all four TODOs in **Part 4**, and check the output matches the expected shape.
2. Complete TODOs **5a–5d** — your agent should settle on `threshold = 3` on its own.
3. Complete TODOs **6a–6c** — find the planted bug and correct the count to 9.
4. Write **both verification notes** in the cells below.
5. Save the notebook and submit the `.ipynb` file on Canvas.

In [ ]:
# ---- Verification note 1: your own output (Part 4) ----
verification_note = """
TODO: In 2-3 sentences, explain how you checked that max(x) is correct
without using max(). What did you check by hand?
"""

# ---- Verification note 2: someone else's output (Parts 5-6) ----
assistant_note = """
TODO: In 2-3 sentences: how did you confirm your Part 5 agent stopped for the
right reason rather than by luck? And what convinced you the Part 6 bug was
really a bug rather than your own misreading?
"""

print(verification_note)
print(assistant_note)